# 🏍️ Notebook 1: The Sidecar Pattern

A **sidecar** is a helper process that runs *next to* your application and adds
cross-cutting features (logging, TLS, metrics, auth) **without touching the app code**.

### Analogy
A motorbike sidecar carries the groceries so the driver can focus on driving.

This is the foundation of service meshes like Istio / Linkerd, where each pod has a proxy sidecar.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Without a sidecar: the app does everything

In [ ]:
import time

def monolithic_handler(request):
    # auth
    if request.get('token') != 'secret': return {'status': 401}
    # logging
    print(f"[log] {request['method']} {request['path']}")
    # metric
    t0 = time.time()
    # actual business logic
    body = {'hello': request.get('name', 'world')}
    print(f'[metric] duration={time.time()-t0:.4f}s')
    return {'status': 200, 'body': body}

print(monolithic_handler({'method':'GET','path':'/hi','token':'secret','name':'ada'}))


Every service must re-implement auth, logging, metrics — in every language. Painful.

## With a sidecar: the app does only business logic

In [ ]:
def business_handler(request):
    """Pure business code — no auth, no logging, no metrics."""
    return {'status': 200, 'body': {'hello': request.get('name', 'world')}}

class Sidecar:
    """Intercepts every request, adds cross-cutting concerns, calls the app."""
    def __init__(self, app):
        self.app = app
    def handle(self, request):
        if request.get('token') != 'secret':
            return {'status': 401}
        print(f"[sidecar log] {request['method']} {request['path']}")
        t0 = time.time()
        resp = self.app(request)
        print(f'[sidecar metric] duration={time.time()-t0:.4f}s')
        return resp

sc = Sidecar(business_handler)
print(sc.handle({'method':'GET','path':'/hi','token':'secret','name':'ada'}))
print(sc.handle({'method':'GET','path':'/hi','token':'bad','name':'ada'}))


The app code shrank. The same sidecar can be reused by every service, in every language.